<center>

# Kappa Foundation — Data Retrieval

***BFIN WS2026 · Investments · Group Assignment***

**Group members:** Võ Hoàng Ngọc Phú (106240420) · Trần Bình Minh (106240415) · Hoàng Trọng Đăng (103240490)

Notebook 01 of 02 · Initial decision record, 22 October 2026
</center>

---

### What this notebook does

* We download every external input used in `02_kappa_analysis.ipynb` and save it **untransformed** in `data/raw/`.
    * Yahoo Finance (via `yfinance`): daily prices, adjusted prices and corporate actions for the candidate universe and the benchmark.
    * Kenneth French Data Library: monthly Fama/French 3 factors and the momentum factor (original ZIP files).
    * U.S. Treasury: daily nominal and real (TIPS) par yield curves, 2015–2022 only.
    * SEC EDGAR: filing indexes used as dated evidence that each security was an SEC-registered, exchange-listed issuer before the case date.
    * Damodaran (NYU Stern): implied equity risk premium workbook dated January 2023, used only as a cross-check of our ERP assumption.
* `data/raw/source_record.csv` logs source, identifier, requested dates, settings, retrieval time (UTC), file, SHA-256 hash and status.

### How to run

* `REDOWNLOAD = False` (default): if a raw file already exists, it is **not** replaced; the cell only re-hashes it. This keeps the frozen 22 Oct data intact and lets *Run All* finish without internet.
* `REDOWNLOAD = True`: downloads again into a **new dated folder** `data/raw_redownload_YYYYMMDD/` so that revised Yahoo/French data can be compared with the frozen files instead of silently replacing them.

---

### Step 1: Settings, universe and download log

All tickers, date ranges and download options are set here.

Market data cover 2015-01-01 to 2025-12-31 because Jan 2023–Dec 2025 is the evaluation period. The analysis notebook only uses data up to 31 Dec 2022 for the decision. Treasury curves stop at 2022, and SEC filings after 2022 are dropped in the analysis.

In [1]:
import json, hashlib, time, io, zipfile, platform, datetime as dt, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, yfinance as yf
import logging
logging.getLogger('yfinance').setLevel(logging.CRITICAL)  # failures are logged in the source record instead
from IPython.display import display, Markdown

REDOWNLOAD = False

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists() and (p / 'notebooks').exists())
RAW_FROZEN = ROOT / 'data' / 'raw'
RAW = RAW_FROZEN if not REDOWNLOAD else ROOT / 'data' / f"raw_redownload_{dt.date.today():%Y%m%d}"
for sub in ['yahoo', 'french', 'treasury', 'sec', 'damodaran']:
    (RAW / sub).mkdir(parents=True, exist_ok=True)

# Contact string required by SEC EDGAR's fair-access policy (replace with a group member's e-mail if needed).
USER_AGENT = 'BFIN-WS2026 Kappa group student project 106240420@student.vgu.edu.vn'

# Candidate universe (team selection) + benchmark. CSGS was screened but is no longer served by Yahoo (see Step 2).
STOCKS = ['MSA', 'KAI', 'GEF', 'IOSP', 'FLO', 'CALM', 'IDA', 'NWN', 'CHE', 'NSIT']
ETFS = ['XMLV', 'RWL']
BENCH = ['SPY']
SCREENED_BUT_FAILED = ['CSGS']
YAHOO_START, YAHOO_END_EXCLUSIVE = '2015-01-01', '2026-01-01'
YF_SETTINGS = dict(interval='1d', auto_adjust=False, actions=True, progress=False, multi_level_index=False)

FRENCH_FILES = {'FF3_monthly': 'F-F_Research_Data_Factors_CSV.zip', 'MOM_monthly': 'F-F_Momentum_Factor_CSV.zip'}
FRENCH_BASE = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
TREASURY_YEARS = range(2015, 2023)   # stops at 2022 = information cutoff
TREASURY_TYPES = {'nominal': 'daily_treasury_yield_curve', 'real': 'daily_treasury_real_yield_curve'}
TREASURY_URL = ('https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/'
                '{y}/all?type={t}&field_tdr_date_value={y}&page&_format=csv')
SEC_CIK = {'MSA': 66570, 'KAI': 886346, 'GEF': 43920, 'IOSP': 1054905, 'FLO': 1128928, 'CALM': 16160,
           'IDA': 1057877, 'NWN': 1733998, 'CHE': 19584, 'NSIT': 932696,
           'XMLV+RWL (Invesco Exchange-Traded Fund Trust II)': 1378872, 'SPY (SPDR S&P 500 ETF Trust)': 884394}
DAMODARAN_URL = 'https://pages.stern.nyu.edu/~adamodar/pc/implprem/ERPJan23.xlsx'

VERSIONS = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'yfinance': yf.__version__}
LOG = []

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def log(source, identifier, request, settings, path, status, rows=None, note=''):
    LOG.append(dict(source=source, identifier=identifier, request=request, settings=json.dumps(settings),
                    retrieved_utc=dt.datetime.fromtimestamp(Path(path).stat().st_mtime, dt.timezone.utc).strftime('%Y-%m-%d %H:%M:%S') if path and Path(path).exists() else '',
                    file=Path(path).relative_to(ROOT).as_posix() if path else '', sha256=sha256(path) if path and Path(path).exists() else '',
                    rows=rows, status=status, note=note, packages=json.dumps(VERSIONS)))

def http_get(url, path, tries=3):
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
            Path(path).write_bytes(urllib.request.urlopen(req, timeout=60).read())
            return True
        except Exception as e:
            err = e; time.sleep(2 * (k + 1))
    print('FAILED', url, err); return False

print('Project root:', ROOT); print('Raw folder  :', RAW); print(VERSIONS)

Project root: /data/kappa
Raw folder  : /data/kappa/data/raw
{'python': '3.13.14', 'pandas': '3.0.6', 'numpy': '2.5.4', 'yfinance': '1.7.0'}


**Output:** raw-data folder and package versions (also written to every row of the source record).

---

### Step 2: Yahoo Finance (daily, unadjusted and adjusted, with actions)

We save the raw `yf.download` response per ticker: `Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits` (ETFs also `Capital Gains`).

We keep daily data with `auto_adjust=False`, so both `Close` (price only) and `Adj Close` (dividends + splits) are available. Month-end values are taken in the analysis.

Each ticker is tried three times; failures are logged, not filled.

In [2]:
def get_yahoo(ticker):
    path = RAW / 'yahoo' / f'{ticker}_daily_raw.csv'
    req = f'{YAHOO_START} to {YAHOO_END_EXCLUSIVE} (end exclusive)'
    if path.exists() and not REDOWNLOAD:
        log('Yahoo Finance via yfinance', ticker, req, YF_SETTINGS, path, 'existing file kept (hash re-checked)', len(pd.read_csv(path)))
        return
    for k in range(3):
        try:
            df = yf.download(ticker, start=YAHOO_START, end=YAHOO_END_EXCLUSIVE, **YF_SETTINGS)
            if len(df):
                df.to_csv(path)
                log('Yahoo Finance via yfinance', ticker, req, YF_SETTINGS, path, 'downloaded', len(df)); return
        except Exception as e:
            pass
        time.sleep(2 * (k + 1))
    log('Yahoo Finance via yfinance', ticker, req, YF_SETTINGS, None, 'FAILED: no data returned',
        note='Excluded from the universe; replaced by another candidate (see decision log).')

for t in STOCKS + ETFS + BENCH + SCREENED_BUT_FAILED:
    get_yahoo(t)
pd.DataFrame(LOG)[['identifier', 'status', 'rows', 'file']]

,identifier,status,rows,file
0,MSA,existing file kept (hash re-checked),2766.0,data/raw/yahoo/MSA_daily_raw.csv
1,KAI,existing file kept (hash re-checked),2766.0,data/raw/yahoo/KAI_daily_raw.csv
2,GEF,existing file kept (hash re-checked),2766.0,data/raw/yahoo/GEF_daily_raw.csv
3,IOSP,existing file kept (hash re-checked),2766.0,data/raw/yahoo/IOSP_daily_raw.csv
4,FLO,existing file kept (hash re-checked),2766.0,data/raw/yahoo/FLO_daily_raw.csv
5,CALM,existing file kept (hash re-checked),2766.0,data/raw/yahoo/CALM_daily_raw.csv
6,IDA,existing file kept (hash re-checked),2766.0,data/raw/yahoo/IDA_daily_raw.csv
7,NWN,existing file kept (hash re-checked),2766.0,data/raw/yahoo/NWN_daily_raw.csv
8,CHE,existing file kept (hash re-checked),2766.0,data/raw/yahoo/CHE_daily_raw.csv
9,NSIT,existing file kept (hash re-checked),2766.0,data/raw/yahoo/NSIT_daily_raw.csv


**Output:** one row per ticker. `CSGS` (CSG Systems, our first IT candidate) returns no data because it is no longer listed — so a universe built only from tickers that still exist today would be biased. We replaced it with `NSIT` (see decision log).

---

### Step 3: Kenneth French Data Library (original ZIP files)

Saved exactly as published; parsing (percent → decimal, monthly block only) is done in the analysis notebook.

| Key | File | Used for |
|---|---|---|
| FF3_monthly | `F-F_Research_Data_Factors_CSV.zip` | Mkt-RF, SMB, HML, RF (CAPM, FF3, risk-free rate) |
| MOM_monthly | `F-F_Momentum_Factor_CSV.zip` | Mom (Carhart 4-factor) |

In [3]:
for key, fname in FRENCH_FILES.items():
    path = RAW / 'french' / fname
    if path.exists() and not REDOWNLOAD:
        log('Kenneth French Data Library', key, FRENCH_BASE + fname, {'format': 'zip/csv'}, path, 'existing file kept (hash re-checked)')
        continue
    ok = http_get(FRENCH_BASE + fname, path)
    log('Kenneth French Data Library', key, FRENCH_BASE + fname, {'format': 'zip/csv'}, path if ok else None,
        'downloaded' if ok else 'FAILED')
    if ok:
        with zipfile.ZipFile(path) as z:
            print(fname, '->', z.namelist())
pd.DataFrame(LOG)[['identifier', 'status', 'file']].tail(2)

,identifier,status,file
14,FF3_monthly,existing file kept (hash re-checked),data/raw/french/F-F_Research_Data_Factors_CSV.zip
15,MOM_monthly,existing file kept (hash re-checked),data/raw/french/F-F_Momentum_Factor_CSV.zip


**Output:** each ZIP contains one CSV, read directly from the ZIP later.

---

### Step 4: U.S. Treasury yield curves, 2015–2022

We need two things from Treasury data:

* **Inflation assumption:** breakeven inflation = nominal par yield − TIPS real par yield (5y and 10y) on 30 Dec 2022.
* **Size of the rate scenarios:** how much the 5y yield and the 2s10s slope moved over 12 months in 2015–2022.

We first tried FRED (`T5YIE`, `DGS5`), but the FRED CSV endpoint returned a maintenance page on our retrieval day, so we use the official Treasury files instead (one CSV per year and curve type).

In [4]:
for kind, tcode in TREASURY_TYPES.items():
    for y in TREASURY_YEARS:
        url = TREASURY_URL.format(y=y, t=tcode)
        path = RAW / 'treasury' / f'{kind}_par_yield_{y}.csv'
        if path.exists() and not REDOWNLOAD:
            log('U.S. Department of the Treasury', f'{kind} par yield curve {y}', url, {'type': tcode}, path,
                'existing file kept (hash re-checked)', len(pd.read_csv(path)))
            continue
        ok = http_get(url, path) and path.read_text()[:4] == 'Date'
        log('U.S. Department of the Treasury', f'{kind} par yield curve {y}', url, {'type': tcode}, path if ok else None,
            'downloaded' if ok else 'FAILED', len(pd.read_csv(path)) if ok else None)
pd.DataFrame(LOG)[['identifier', 'status', 'rows']].tail(16)

,identifier,status,rows
16,nominal par yield curve 2015,existing file kept (hash re-checked),251.0
17,nominal par yield curve 2016,existing file kept (hash re-checked),250.0
18,nominal par yield curve 2017,existing file kept (hash re-checked),250.0
19,nominal par yield curve 2018,existing file kept (hash re-checked),249.0
20,nominal par yield curve 2019,existing file kept (hash re-checked),250.0
21,nominal par yield curve 2020,existing file kept (hash re-checked),251.0
22,nominal par yield curve 2021,existing file kept (hash re-checked),251.0
23,nominal par yield curve 2022,existing file kept (hash re-checked),249.0
24,real par yield curve 2015,existing file kept (hash re-checked),251.0
25,real par yield curve 2016,existing file kept (hash re-checked),250.0


**Output:** 16 files (8 years × nominal/real), each checked to start with a `Date` header.

---

### Step 5: SEC EDGAR filing indexes (eligibility evidence)

The brief asks for evidence that each security was eligible at the case date. EDGAR lists every filing with its date, so a 10-Q/10-K (stocks) or 485BPOS/N-CSR (ETF trust) filed **before 31 Dec 2022** shows the issuer was registered and reporting at that time.

Older filings sit in extra pages (`filings.files`); we save all pages.

In [5]:
for label, cik in SEC_CIK.items():
    base = f'https://data.sec.gov/submissions/CIK{cik:010d}.json'
    path = RAW / 'sec' / f'CIK{cik:010d}.json'
    if path.exists() and not REDOWNLOAD:
        log('SEC EDGAR submissions API', label, base, {'User-Agent': 'see notebook'}, path, 'existing file kept (hash re-checked)')
    else:
        ok = http_get(base, path); time.sleep(0.3)
        log('SEC EDGAR submissions API', label, base, {'User-Agent': 'see notebook'}, path if ok else None, 'downloaded' if ok else 'FAILED')
    if path.exists():
        for extra in json.loads(path.read_text())['filings'].get('files', []):
            p2 = RAW / 'sec' / extra['name']
            if p2.exists() and not REDOWNLOAD:
                log('SEC EDGAR submissions API', label + ' (older filings page)', 'https://data.sec.gov/submissions/' + extra['name'],
                    {}, p2, 'existing file kept (hash re-checked)')
                continue
            ok = http_get('https://data.sec.gov/submissions/' + extra['name'], p2); time.sleep(0.3)
            log('SEC EDGAR submissions API', label + ' (older filings page)', 'https://data.sec.gov/submissions/' + extra['name'],
                {}, p2 if ok else None, 'downloaded' if ok else 'FAILED')
print(sorted(p.name for p in (RAW / 'sec').glob('*.json')))

['CIK0000016160-submissions-001.json', 'CIK0000016160.json', 'CIK0000019584-submissions-001.json', 'CIK0000019584.json', 'CIK0000043920-submissions-001.json', 'CIK0000043920.json', 'CIK0000066570-submissions-001.json', 'CIK0000066570.json', 'CIK0000884394.json', 'CIK0000886346-submissions-001.json', 'CIK0000886346.json', 'CIK0000932696-submissions-001.json', 'CIK0000932696.json', 'CIK0001054905-submissions-001.json', 'CIK0001054905.json', 'CIK0001057877-submissions-001.json', 'CIK0001057877.json', 'CIK0001128928-submissions-001.json', 'CIK0001128928.json', 'CIK0001378872-submissions-001.json', 'CIK0001378872-submissions-002.json', 'CIK0001378872-submissions-003.json', 'CIK0001378872.json', 'CIK0001733998.json']


**Output:** saved JSON pages. The same filings can be checked in a browser at
`https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=<CIK>&type=10-Q&dateb=20221231`.

---

### Step 6: Damodaran implied ERP workbook (January 2023 update)

Only a cross-check. The workbook uses the S&P 500 level at the start of January 2023 but was published a few days after our cutoff, so we set our ERP (5.0%) ourselves and only compare it with this number.

In [6]:
path = RAW / 'damodaran' / 'ERPJan23.xlsx'
if path.exists() and not REDOWNLOAD:
    log('Damodaran Online (NYU Stern)', 'ERPJan23.xlsx', DAMODARAN_URL, {}, path, 'existing file kept (hash re-checked)')
else:
    ok = http_get(DAMODARAN_URL, path)
    log('Damodaran Online (NYU Stern)', 'ERPJan23.xlsx', DAMODARAN_URL, {}, path if ok else None, 'downloaded' if ok else 'FAILED')
print(path.exists(), path.stat().st_size if path.exists() else None)

True 1497908


---

### Step 7: Source record

On a re-run the original record (with the first retrieval times) is kept and only the hashes are checked.

In [7]:
new = pd.DataFrame(LOG)
rec_path = RAW / 'source_record.csv'
if rec_path.exists() and not REDOWNLOAD:
    old = pd.read_csv(rec_path, dtype=str, keep_default_na=False)
    merged = old.copy()
    for _, r in new.iterrows():
        m = (old['identifier'] == str(r['identifier'])) & (old['file'].fillna('') == str(r['file']))
        if m.any():
            idx = old.index[m][0]
            if str(old.at[idx, 'sha256']) != str(r['sha256'] or ''):
                raise ValueError(f"Hash mismatch for {r['file'] or r['identifier']} - raw file changed!")
        else:
            merged = pd.concat([merged, pd.DataFrame([r]).astype(str)], ignore_index=True)
    record = merged
    if len(merged) > len(old):
        record.to_csv(rec_path, index=False)
    print('Existing source record kept; all hashes match the saved files.')
else:
    record = new
    record['status'] = record['status'].replace('existing file kept (hash re-checked)', 'downloaded')
    record.to_csv(rec_path, index=False)
    print('Source record written:', rec_path.relative_to(ROOT))
summary = record.groupby(['source', 'status']).size().rename('files').reset_index()
display(summary)
failed = record[record['status'].astype(str).str.startswith('FAILED')]
display(Markdown(f"**Failed downloads:** {', '.join(failed['identifier']) if len(failed) else 'none'} — documented, not filled."))

Existing source record kept; all hashes match the saved files.


,source,status,files
0,Damodaran Online (NYU Stern),downloaded,1
1,Kenneth French Data Library,downloaded,2
2,SEC EDGAR submissions API,downloaded,24
3,U.S. Department of the Treasury,downloaded,16
4,Yahoo Finance via yfinance,FAILED: no data returned,1
5,Yahoo Finance via yfinance,downloaded,13


**Failed downloads:** CSGS — documented, not filled.

**Output:** every input file has a hash. `02_kappa_analysis.ipynb` checks these hashes first and stops if any file changed.

---